In [3]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os

def rasterize_by_grid_with_auto_class_mapping(shapefile_path, grid_path, class_field, output_dir, resolution=10, nodata=255):
    # 读取数据
    gdf = gpd.read_file(shapefile_path)
    grid = gpd.read_file(grid_path)

    # 坐标系统统一为投影坐标系（UTM 51N）
    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(epsg=32651)
    if gdf.crs != grid.crs:
        grid = grid.to_crs(gdf.crs)

    # 清理无效几何
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    # 检查字段是否存在
    if class_field not in gdf.columns:
        raise ValueError(f"字段 {class_field} 不存在于 shapefile 中")

    # 获取字段的唯一值列表，并生成映射字典（类别字符串->数字ID，从1开始，0 留作背景）
    unique_classes = sorted(gdf[class_field].dropna().unique())
    class_to_id = {cls: idx+1 for idx, cls in enumerate(unique_classes)}
    print(f"自动生成类别映射（{class_field}）：{class_to_id}")

    # 用映射替换原字段，未包含的赋值0
    gdf['class_val'] = gdf[class_field].map(class_to_id).fillna(0).astype(int)

    os.makedirs(output_dir, exist_ok=True)

    for idx, cell in grid.iterrows():
        grid_geom = cell.geometry

        clipped = gpd.overlay(gdf, gpd.GeoDataFrame(geometry=[grid_geom], crs=gdf.crs), how="intersection")
        if clipped.empty:
            continue

        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        shapes = [(geom, val) for geom, val in zip(clipped.geometry, clipped['class_val'])]

        raster = rasterize(
            shapes=shapes,
            out_shape=(height, width),
            transform=transform,
            fill=nodata,
            dtype='uint8'
        )
        tile_id = cell["ID"]
        out_path = os.path.join(output_dir, f"tile_{tile_id}.tif")

        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="uint8",
            crs=gdf.crs,
            transform=transform,
            nodata=nodata
        ) as dst:
            dst.write(raster, 1)

        print(f"✅ 输出: {out_path}")

# 使用示例
rasterize_by_grid_with_auto_class_mapping(
    shapefile_path=r"D:\shanghai workshop\vector\Shanghai street_ver3\Shanghai street_ver3.shp",
    grid_path=r"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp",
    class_field="Type",
    output_dir=r"D:\shanghai workshop\raster\final_street_tiles",
    resolution=1,
    nodata=255
)

自动生成类别映射（Type）：{'中心城区主要道路': 1, '中心城区道路': 2, '主干道路': 3, '公路': 4, '快速路': 5, '快速路匝道': 6, '支路': 7, '郊县中心城区主干道路': 8, '高速': 9, '高速公路匝道': 10}
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_192.tif
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_193.tif
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_196.tif
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_532.tif
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_533.tif
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_534.tif
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_535.tif
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_536.tif
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_539.tif
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_874.tif
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_875.tif
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_876.tif
✅ 输出: D:\shanghai workshop\raster\final_street_tiles\tile_878.tif
✅ 输出: D

In [8]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os
import numpy as np

def rasterize_multiple_fields_to_multiband(shapefile_path, grid_path, class_fields, output_dir, resolution=10, nodata=255, start_id=0):
    gdf = gpd.read_file(shapefile_path)
    grid = gpd.read_file(grid_path)

    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(epsg=32651)
    if gdf.crs != grid.crs:
        grid = grid.to_crs(gdf.crs)

    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    os.makedirs(output_dir, exist_ok=True)

    # ✅ 第一步：提前为所有字段生成全局映射字典
    field_maps = {}
    for field in class_fields:
        if field not in gdf.columns:
            raise ValueError(f"字段 {field} 不存在于 shapefile 中")
        if gdf[field].dtype.kind in ['O', 'U']:  # 字符型
            unique_vals = sorted(gdf[field].dropna().unique())
            val_map = {v: i+1 for i, v in enumerate(unique_vals)}
            print(f"字段 {field} 的映射：{val_map}")
            field_maps[field] = val_map
            gdf[f"__mapped_{field}"] = gdf[field].map(val_map).fillna(0).astype(int)
        else:
            gdf[f"__mapped_{field}"] = gdf[field].fillna(0)

    for idx, cell in grid.iterrows():
        tile_id = cell["ID"]

        # ✅ 跳过起始 ID 之前的 tile
        if tile_id < start_id:
            continue

        out_path = os.path.join(output_dir, f"building_tile_{tile_id}.tif")

        if os.path.exists(out_path):
            print(f"⏩ 已存在，跳过: {out_path}")
            continue

        grid_geom = cell.geometry
        clipped = gpd.overlay(gdf, gpd.GeoDataFrame(geometry=[grid_geom], crs=gdf.crs), how="intersection")
        if clipped.empty:
            continue

        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        bands = []

        for field in class_fields:
            mapped_col = f"__mapped_{field}"
            class_vals = clipped[mapped_col]
            shapes = [(geom, val) for geom, val in zip(clipped.geometry, class_vals)]

            raster = rasterize(
                shapes=shapes,
                out_shape=(height, width),
                transform=transform,
                fill=nodata,
                dtype='uint8' if class_vals.dtype == int else 'float32'
            )
            bands.append(raster)

        bands_array = np.stack(bands, axis=0)

        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=len(class_fields),
            dtype=bands_array.dtype,
            crs=gdf.crs,
            transform=transform,
            nodata=nodata
        ) as dst:
            dst.write(bands_array)

        print(f"✅ 输出: {out_path}")

rasterize_multiple_fields_to_multiband(
    shapefile_path=r'D:\shanghai workshop\vector\Building_processed nearest 1 to 500m_finver\result\building_UTM51N.shp',
    grid_path=r"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp",
    class_fields=["Height", "Function", "Style"],
    output_dir=r"D:\shanghai workshop\raster\final_building_tiles_with_bands",
    resolution=1,
    nodata=255,
    start_id=400
)

字段 Function 的映射：{'Business': 1, 'Industry': 2, 'Office': 3, 'Other': 4, 'Public service': 5, 'Residence': 6}
字段 Style 的映射：{'M': 1, 'Modern': 2, 'Postmodern': 3, 'Traditional': 4}
⏩ 已存在，跳过: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_532.tif
⏩ 已存在，跳过: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_534.tif
⏩ 已存在，跳过: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_535.tif
⏩ 已存在，跳过: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_536.tif
⏩ 已存在，跳过: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_874.tif
⏩ 已存在，跳过: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_875.tif
⏩ 已存在，跳过: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_1217.tif
⏩ 已存在，跳过: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_1936.tif
⏩ 已存在，跳过: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile